# Ultralytics Classifier for Video Frame Classification

This notebook demonstrates how to use the Ultralytics Classifier to perform classification on video frames stored in a local directory.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
from footy_track.classifier import UltralyticsClassifier
from footy_track import constants
import fiftyone as fo
from tqdm.auto import tqdm


import os

In [ ]:

print(fo.list_datasets())
for ds in fo.list_datasets():
    fo.delete_dataset(ds)
    print(f"{ds=} deleted")

## Define Input Frame Directory

Specify the path to the directory containing the video frames to be classified.

In [ ]:
FRAMES_DIR = Path("/Users/georgebarnett/code/football-scan/data/arsenal_mancity/full_video_frames")

if not FRAMES_DIR.exists():
    raise FileNotFoundError(f"Frames directory not found: {FRAMES_DIR}")

print(f"Processing frames from: {FRAMES_DIR}")

## Load the Ultralytics Classifier

Initialize and load the pre-trained Ultralytics classifier model.

In [ ]:
classifier = UltralyticsClassifier(model_path="/Users/georgebarnett/code/football-scan/model_saves/classifier/20251226-yolo11n-cls/0.987.pt")
print("Ultralytics Classifier loaded.")

## Run Inference on Frames

Iterate through each image file in the defined input directory and perform classification.

In [ ]:
all_image_files = sorted(list(FRAMES_DIR.glob(f"*.{constants.IMAGE_FORMAT}")))

# NUM_SAMPLES = 1000
NUM_SAMPLES = None
if NUM_SAMPLES is None:
    image_paths = all_image_files
else:
    # Sample NUM_SAMPLES images evenly from the entire list of images
    total_files = len(all_image_files)
    if NUM_SAMPLES > total_files:
        image_paths = all_image_files
    else:
        # Generate evenly spaced indices
        indices = [round(i * (total_files - 1) / (NUM_SAMPLES - 1)) for i in range(NUM_SAMPLES)]
        # Use a set to remove potential duplicate indices from rounding, then sort
        unique_indices = sorted(list(set(indices)))
        image_paths = [all_image_files[i] for i in unique_indices]

print(f"{len(image_paths)=}")

In [ ]:
dataset_name = "mancity_arsenal_visualisations"
if fo.dataset_exists(dataset_name):
    fo.delete_dataset(dataset_name)
dataset = fo.Dataset(dataset_name)

print("Running prediction")
samples = []
for i, image_path in enumerate(tqdm(image_paths)):
    result = classifier.predict_from_path(image_path)
    samples.append(
        result.to_fiftyone_sample()
    )

print("Adding to dataset")
dataset.add_samples(samples)
len(dataset)

## Compute Embeddings and Visualize

To visualize the dataset in 2D, we first need to compute embeddings for each image. We'll use a pre-trained model from the FiftyOne Model Zoo to generate these embeddings. After that, we can use `fiftyone.brain` to compute a 2D visualization of the data.

In [ ]:
import fiftyone.zoo as foz
from fiftyone import brain

# Generate embeddings
embedding_model = foz.load_zoo_model("mobilenet-v2-imagenet-torch", device="mps")
embeddings = dataset.compute_embeddings(embedding_model)

# Compute visualization
results = brain.compute_visualization(dataset, embeddings=embeddings, brain_key="img_viz")

print("Embeddings computed and visualization is ready.")


## Display Results

Visualize the classification results using FiftyOne.

In [ ]:
session = fo.launch_app(dataset)

## Get Selected Frames

After selecting frames in the FiftyOne app, run the cell below to get the filepaths of the selected frames.

In [ ]:
selected_frames = []


In [ ]:

selected_samples = session.selected
if not selected_samples:
    print("No samples selected in the FiftyOne app.")
else:
    print(f"{len(selected_samples)} samples selected.")
    
    # Create a view of the selected samples
    selected_view = dataset.select(selected_samples)
    
    # Get the filepaths of the selected samples
    selected_filepaths = selected_view.values("filepath")
    
    print("Filepaths of selected frames:")
    for filepath in selected_filepaths:
        selected_frames.append(filepath)


selected_filepaths = list(set(selected_filepaths))
print(len(selected_filepaths))
selected_filepaths


## Upload Selected Frames to Roboflow

Now, we'll use the `RoboflowClassificationHandler` to upload the selected frames to your Roboflow project. Make sure your `ROBOFLOW_API_KEY` is set as an environment variable.

In [ ]:
from footy_track.labelling import RoboflowClassificationHandler
from footy_track import constants

# Roboflow configuration
batch_name = "uploading incorrect for training"

# Initialize the handler
handler = RoboflowClassificationHandler(
    workspace_name=constants.ROBOFLOW_WORKSPACE, 
    project_name=constants.ROBOFLOW_BROADCAST_PROJECT,
    classifier=classifier,
)

# Upload the images
if selected_filepaths:
    print(f"Uploading {len(selected_filepaths)} images to Roboflow...")
    handler.upload_images(image_paths=[Path(p) for p in selected_filepaths], batch_name=batch_name)
    print("Upload complete.")
else:
    print("No images to upload.")